In [0]:
# Uber Trip Analytics - Data Preprocessing

## Capstone Project

### Project Objective
This project aims to build an end-to-end Data Engineering and Machine Learning pipeline using Uber trip data.

### Technologies Used
- AWS S3
- Databricks
- PySpark
- Delta Lake
- dbt
- Amazon Redshift
- Tableau
- Docker
- Kubernetes
- GitHub

### Dataset
Uber Trips Dataset (50K Records)

### Notebook
01_Data_Preprocessing

  File <command-4975079839076748>, line 22
    Uber Trips Dataset (50K Records)
                         ^
SyntaxError: invalid decimal literal


In [0]:
# ============================================
# Import Required Libraries
# ============================================

import pandas as pd

from pyspark.sql.functions import (
    col,
    when,
    hour,
    dayofweek,
    month,
    to_timestamp,
    round,
    expr,
    count,
    isnan
)

from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler,
    StandardScaler
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [0]:
# ============================================
# Load Uber Dataset
# ============================================

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("dbfs:/Volumes/workspace/default/uber_data/uber_trips_dataset_50k.csv")

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [0]:
display(df.limit(10))

trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time
1,8270,10683,San Francisco,37.17093115042939,-77.58647936979322,37.1736523397383,-77.61993433646886,2.97,10.71,Completed,Wallet,2023-01-01T00:00:00.000Z,2023-01-01T00:08:54.600Z
2,1860,44743,Boston,38.89812691856606,-108.58297695484846,38.93746379536704,-108.55872717079913,8.43,22.41,Completed,UPI,2023-01-01T00:01:00.000Z,2023-01-01T00:26:17.400Z
3,6390,75839,San Francisco,38.81457056869823,-89.94260270822319,38.82170236125605,-89.8964345566054,5.46,12.91,Completed,Cash,2023-01-01T00:02:00.000Z,2023-01-01T00:18:22.800Z
4,6191,22189,New York,37.29590568598714,-75.32884414927358,37.30137520518609,-75.31748783882725,6.61,15.7,Completed,Wallet,2023-01-01T00:03:00.000Z,2023-01-01T00:22:49.800Z
5,6734,61104,Seattle,38.97239533578873,-121.48291286204801,38.99208841336341,-121.46790441380276,10.5,19.15,Completed,Wallet,2023-01-01T00:04:00.000Z,2023-01-01T00:35:30.000Z
6,7265,84988,San Francisco,37.600550944815616,-106.71906865688572,37.55691580938672,-106.73095912437086,9.94,19.95,Completed,Card,2023-01-01T00:05:00.000Z,2023-01-01T00:34:49.200Z
7,1466,82933,San Francisco,40.28855964243779,-82.8082193858109,40.27883011229549,-82.84463502060092,12.22,25.89,Completed,UPI,2023-01-01T00:06:00.000Z,2023-01-01T00:42:39.600Z
8,5426,60182,Chicago,40.601188277022246,-78.39873323235332,40.569810860072174,-78.36818593976587,10.14,25.55,Completed,Cash,2023-01-01T00:07:00.000Z,2023-01-01T00:37:25.200Z
9,6578,15826,San Francisco,40.29624711069893,-87.57651655291244,40.284637585308246,-87.56478285126917,1.88,7.97,Completed,Cash,2023-01-01T00:08:00.000Z,2023-01-01T00:13:38.400Z
10,9322,63503,Boston,40.71263418106593,-78.49965570372784,40.67026654819631,-78.477475359972,3.7,12.21,No-Show,UPI,2023-01-01T00:09:00.000Z,2023-01-01T00:20:06.000Z


In [0]:
df.printSchema()

root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- rider_id: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- pickup_lat: double (nullable = true)
 |-- pickup_lng: double (nullable = true)
 |-- drop_lat: double (nullable = true)
 |-- drop_lng: double (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- drop_time: timestamp (nullable = true)



In [0]:
print("="*50)
print("Dataset Information")
print("="*50)

print("Rows :", df.count())
print("Columns :", len(df.columns))

Dataset Information
Rows : 50000
Columns : 14


In [0]:
print("="*50)
print("Column Names")
print("="*50)

for column in df.columns:
    print(column)

Column Names
trip_id
driver_id
rider_id
city
pickup_lat
pickup_lng
drop_lat
drop_lng
distance_km
fare_amount
status
payment_method
pickup_time
drop_time


In [0]:
display(df.describe())

summary,trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method
count,50000,50000,50000,50000,50000,50000,50000,50000,50000,50000,50000,50000
mean,25000.5,5493.95454,55040.99458,null,38.99866366878846,-97.4855516351905,38.99876835433958,-97.48567722390504,7.0080702000000095,15.976205400000007,null,null
stddev,14433.90106658626,2601.410798147095,25915.467105374,null,1.1552411010088783,14.173744187393378,1.1556955706301415,14.173734666462437,2.9467441347853525,6.274406063241348,null,null
min,1,1000,10001,Boston,37.000009105082164,-121.99946522741315,36.953364823297214,-122.04712350229215,0.0,1.08,Cancelled,Card
max,50000,9998,99998,Seattle,40.99993677379934,-73.00196097873916,41.04753634317474,-72.96347649069838,19.41,50.67,No-Show,Wallet


In [0]:
display(df.dtypes)

_1,_2
trip_id,int
driver_id,int
rider_id,int
city,string
pickup_lat,double
pickup_lng,double
drop_lat,double
drop_lng,double
distance_km,double
fare_amount,double


In [0]:
from pyspark.sql.functions import col,isnull,sum

display(
    df.select(
        [
            sum(
                col(c).isNull().cast("int")
            ).alias(c)
            for c in df.columns
        ]
    )
)

trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
print("Total Records :", df.count())

print("Distinct Records :", df.distinct().count())

Total Records : 50000
Distinct Records : 50000


In [0]:
from pyspark.sql.functions import col

total_rows = df.count()

print("="*50)
print("Missing Value Percentage")
print("="*50)

for c in df.columns:

    null_count = df.filter(col(c).isNull()).count()

    percentage = (null_count / total_rows) * 100

    print(f"{c} : {percentage:.2f}%")

Missing Value Percentage
trip_id : 0.00%
driver_id : 0.00%
rider_id : 0.00%
city : 0.00%
pickup_lat : 0.00%
pickup_lng : 0.00%
drop_lat : 0.00%
drop_lng : 0.00%
distance_km : 0.00%
fare_amount : 0.00%
status : 0.00%
payment_method : 0.00%
pickup_time : 0.00%
drop_time : 0.00%


In [0]:
# ==========================================
# Remove Duplicate Records
# ==========================================

before = df.count()

df = df.dropDuplicates()

after = df.count()

print("Rows Before :", before)
print("Rows After  :", after)

Rows Before : 50000
Rows After  : 50000


In [0]:
# ==========================================
# Remove Null Values
# ==========================================

before = df.count()

df = df.dropna()

after = df.count()

print("Rows Before :", before)
print("Rows After  :", after)

Rows Before : 50000
Rows After  : 50000


In [0]:
# ==========================================
# Invalid Distance
# ==========================================

from pyspark.sql.functions import col

display(
    df.filter(col("distance_km") <= 0)
)

trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time
39375,1857,62380,Chicago,37.03084135012183,-77.4444269558719,37.07161442052239,-77.44591346767797,0.0,4.02,Completed,Card,2023-01-28T08:14:00.000Z,2023-01-28T08:14:00.000Z
39362,2853,83128,Seattle,39.934208748427594,-84.13215978992818,39.98247030416181,-84.10325197116431,0.0,3.97,No-Show,UPI,2023-01-28T08:01:00.000Z,2023-01-28T08:01:00.000Z
44303,3552,46859,Seattle,40.36826189352436,-77.43673181255124,40.3504232646916,-77.45848695936243,0.0,2.01,Completed,Card,2023-01-31T18:22:00.000Z,2023-01-31T18:22:00.000Z


In [0]:

#Remove Invalid Distance
df = df.filter(col("distance_km") > 0)

print("Remaining Records :", df.count())

Remaining Records : 49997


In [0]:
#Check Invalid Fare
display(
    df.filter(col("fare_amount") <= 0)
)

trip_id,driver_id,rider_id,city,pickup_lat,pickup_lng,drop_lat,drop_lng,distance_km,fare_amount,status,payment_method,pickup_time,drop_time


In [0]:

#Verify Timestamp Columns
df = df.filter(col("fare_amount") > 0)

print("Remaining Records :", df.count())

Remaining Records : 49997


In [0]:
#Final Clean Dataset Information

print("=" * 50)
print("Clean Dataset Information")
print("=" * 50)

print("Rows :", df.count())
print("Columns :", len(df.columns))


Clean Dataset Information
Rows : 49997
Columns : 14


In [0]:
#Verify Timestamp Columns
display(
    df.select(
        "pickup_time",
        "drop_time"
    ).limit(10)
)

pickup_time,drop_time
2023-01-01T00:06:00.000Z,2023-01-01T00:42:39.600Z
2023-01-01T00:27:00.000Z,2023-01-01T00:43:21.000Z
2023-01-01T00:37:00.000Z,2023-01-01T00:48:04.200Z
2023-01-01T00:44:00.000Z,2023-01-01T01:21:12.000Z
2023-01-01T01:05:00.000Z,2023-01-01T01:43:07.800Z
2023-01-01T01:27:00.000Z,2023-01-01T01:47:54.600Z
2023-01-01T01:51:00.000Z,2023-01-01T02:16:46.200Z
2023-01-01T01:55:00.000Z,2023-01-01T02:06:52.800Z
2023-01-01T02:26:00.000Z,2023-01-01T02:45:40.800Z
2023-01-01T02:48:00.000Z,2023-01-01T03:03:50.400Z


In [0]:
# Save Cleaned Dataset

df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/uber_data/cleaned_dataset"
)

print("Cleaned Dataset Saved Successfully")

Cleaned Dataset Saved Successfully


In [0]:
#cleaned Data
clean_df = spark.read.parquet("/Volumes/workspace/default/uber_data/cleaned_dataset")

clean_df.coalesce(1).write.mode("overwrite") \
.option("header", "true") \
.csv("/Volumes/workspace/default/uber_data/cleaned_data_csv")